# 🤖 ACIS Insurance — Statistical Modeling & Risk-Based Pricing
## Task 4: Building the Pricing Engine
---
**Two models form the pricing engine:**

| Model | Type | Target | Metrics |
|---|---|---|---|
| Claim Severity | Regression | TotalClaims (where > 0) | RMSE, R² |
| Claim Probability | Classification | HasClaim (0/1) | F1, Precision, Recall |

**Three algorithms compared for each:**
Linear Regression → Random Forest → XGBoost

**Combined formula:**
`Premium = P(claim) × Predicted Severity + Expense Loading + Profit Margin`

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import shap

from src.data_loader import load_and_prepare
from src.modeling import (
    engineer_features,
    prepare_features,
    train_severity_models,
    train_probability_models,
    compare_models,
    calculate_premium
)

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ All imports successful')

---
## 1. Data Preparation & Feature Engineering

In [ ]:
# Load data
DATA_PATH = '../data/MachineLearningRating_v3.txt'
df = load_and_prepare(DATA_PATH)

# Add claim flag
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

print(f'Full dataset: {df.shape[0]:,} rows')
print(f'Policies with claims: {df["HasClaim"].sum():,} '
      f'({df["HasClaim"].mean():.2%})')

# Engineer new features
print('\nEngineering features...')
df = engineer_features(df)

print(f'\nFinal dataset: {df.shape[0]:,} rows, '
      f'{df.shape[1]} columns')

---
## 2. Model 1 — Claim Severity (Regression)

**Goal:** For policies that had a claim, predict
how much the claim will cost.

**Why only claims > 0?**
Including zero-claim policies would force the model
to predict near-zero for almost everything —
it would learn nothing about actual claim costs.

**Data subset:** ~2,800 policies (0.28% of portfolio)

In [ ]:
# ─── SEVERITY MODEL PREP ────────────────────────────────────

# Only use policies that had a claim
claims_df = df[df['TotalClaims'] > 0].copy()
print(f'Claim policies for severity model: {len(claims_df):,}')

# Log-transform target
# We saw in EDA that TotalClaims is right-skewed
# Log transform makes it more normal → better model fit
# np.log1p = log(x+1), safe for any positive number
claims_df['LogClaims'] = np.log1p(claims_df['TotalClaims'])

print(f'\nTotalClaims stats (raw):')
print(claims_df['TotalClaims'].describe().round(2))
print(f'\nLogClaims stats (transformed):')
print(claims_df['LogClaims'].describe().round(4))

# Prepare features
print('\nPreparing features...')
X_sev, y_sev, feature_names, encoders = prepare_features(
    claims_df,
    target_col='LogClaims'   # predict log-transformed claims
)

# Split 80/20
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sev, y_sev,
    test_size=0.2,
    random_state=42
)

print(f'\nTrain set: {len(X_train_s):,} policies')
print(f'Test set:  {len(X_test_s):,} policies')

In [ ]:
# ─── TRAIN SEVERITY MODELS ──────────────────────────────────
severity_results = train_severity_models(
    X_train_s, y_train_s,
    X_test_s,  y_test_s
)

In [ ]:
# ─── MODEL COMPARISON TABLE ─────────────────────────────────
sev_comparison = compare_models(severity_results, task='regression')

print('\nCLAIM SEVERITY MODEL COMPARISON')
print('=' * 50)
print(sev_comparison[['Model', 'RMSE', 'R²']].to_string(index=False))
print('\nNote: RMSE is in log-scale (target was log-transformed)')
print('Lower RMSE = better | Higher R² = better')

# Identify best model
best_sev = min(
    severity_results.items(),
    key=lambda x: x[1]['rmse']
)[0]
print(f'\n🏆 Best severity model: {best_sev}')

In [ ]:
# ─── PREDICTED VS ACTUAL PLOT ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Severity Models — Predicted vs Actual',
             fontsize=14, fontweight='bold')

colors = ['#3498db', '#e67e22', '#e74c3c']

for ax, (name, res), color in zip(
    axes, severity_results.items(), colors
):
    y_pred = res['predictions']

    ax.scatter(y_test_s, y_pred,
               alpha=0.3, s=10, color=color)

    # Perfect prediction line
    min_val = min(y_test_s.min(), y_pred.min())
    max_val = max(y_test_s.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val],
            'k--', linewidth=1.5, label='Perfect')

    ax.set_title(f'{name}\nRMSE={res["rmse"]:.3f} '
                 f'R²={res["r2"]:.3f}')
    ax.set_xlabel('Actual Log(Claims)')
    ax.set_ylabel('Predicted Log(Claims)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/severity_predicted_vs_actual.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Model 2 — Claim Probability (Classification)

**Goal:** Predict whether any given policy will
result in a claim at all.

**Class imbalance challenge:**
Only 0.28% of policies have claims.
Without correction, the model would predict
"no claim" for everything and be 99.72% accurate
— but completely useless.

**Solution:** Use `class_weight='balanced'` and
`scale_pos_weight` in XGBoost to give the minority
class (claims) proportionally more weight.

**Key metric: F1 Score (not accuracy)**
F1 balances Precision and Recall:
- Precision: of all predicted claims, how many were real?
- Recall:    of all real claims, how many did we catch?

`F1 = 2 × (Precision × Recall) / (Precision + Recall)`

In [ ]:
# ─── PROBABILITY MODEL PREP ─────────────────────────────────

# Use full dataset — all policies
print('Preparing probability model features...')

X_prob, y_prob_target, feat_names_prob, enc_prob = prepare_features(
    df,
    target_col='HasClaim'
)

# Split 80/20 — stratify keeps class ratio same in both sets
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_prob, y_prob_target,
    test_size=0.2,
    random_state=42,
    stratify=y_prob_target   # keeps 0.28% ratio in both sets
)

print(f'\nTrain set: {len(X_train_p):,} | '
      f'Claims: {y_train_p.sum():,} '
      f'({y_train_p.mean():.2%})')
print(f'Test set:  {len(X_test_p):,} | '
      f'Claims: {y_test_p.sum():,} '
      f'({y_test_p.mean():.2%})')

In [ ]:
# ─── TRAIN PROBABILITY MODELS ───────────────────────────────
prob_results = train_probability_models(
    X_train_p, y_train_p,
    X_test_p,  y_test_p
)

In [ ]:
# ─── CLASSIFICATION COMPARISON TABLE ────────────────────────
prob_comparison = compare_models(
    prob_results, task='classification'
)

print('\nCLAIM PROBABILITY MODEL COMPARISON')
print('=' * 65)
print(prob_comparison[
    ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score']
].to_string(index=False))

best_prob = max(
    prob_results.items(),
    key=lambda x: x[1]['f1']
)[0]
print(f'\n🏆 Best probability model: {best_prob}')

---
## 4. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) explains
WHY the model makes each prediction.

We run SHAP on the best performing severity model
since severity prediction is the core of the
pricing formula.

**Expected outcome based on Task 3 findings:**
If PostalCode and Province appear as top SHAP
features, it validates our hypothesis testing.

In [ ]:
# ─── SHAP ANALYSIS ──────────────────────────────────────────
print('Running SHAP analysis on best severity model...')
print('(This may take 1-2 minutes)')

# Get the best severity model
best_sev_model = severity_results[best_sev]['model']

# SHAP TreeExplainer is fast for tree-based models
# (Random Forest and XGBoost)
# For Linear Regression use LinearExplainer instead
if best_sev in ['Random Forest', 'XGBoost']:
    explainer = shap.TreeExplainer(best_sev_model)
else:
    explainer = shap.LinearExplainer(
        best_sev_model, X_train_s
    )

# Calculate SHAP values on test set
# Use a sample of 500 for speed
sample_idx = np.random.choice(
    len(X_test_s), size=min(500, len(X_test_s)),
    replace=False
)
X_shap_sample = X_test_s.iloc[sample_idx]
shap_values = explainer.shap_values(X_shap_sample)

print('✅ SHAP values calculated')
print(f'Shape: {shap_values.shape}')

In [ ]:
# ─── SHAP SUMMARY PLOT ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Bar chart of mean absolute SHAP values
# Shows which features matter most overall
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature':    feature_names,
    'SHAP_Value': mean_shap
}).sort_values('SHAP_Value', ascending=True).tail(15)

axes[0].barh(
    feature_importance['Feature'],
    feature_importance['SHAP_Value'],
    color='#2E75B6', alpha=0.85
)
axes[0].set_title(
    f'Top Features by SHAP Importance\n({best_sev})',
    fontsize=13, fontweight='bold'
)
axes[0].set_xlabel('Mean |SHAP Value| (impact on log claims)')

# Plot 2: SHAP beeswarm (dot plot)
# Shows direction AND magnitude of each feature's impact
plt.sca(axes[1])
shap.summary_plot(
    shap_values,
    X_shap_sample,
    feature_names=feature_names,
    max_display=15,
    show=False,
    plot_type='dot'
)
axes[1].set_title(
    'SHAP Values — Direction & Magnitude',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('../reports/shap_summary.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── TOP FEATURES TABLE ─────────────────────────────────────
mean_shap_all = np.abs(shap_values).mean(axis=0)
top_features = pd.DataFrame({
    'Feature':   feature_names,
    'Mean_SHAP': mean_shap_all,
}).sort_values('Mean_SHAP', ascending=False).head(10)

top_features['Rank'] = range(1, 11)
top_features['Mean_SHAP'] = top_features['Mean_SHAP'].round(4)

print('TOP 10 FEATURES BY SHAP IMPORTANCE')
print('=' * 50)
print(top_features[['Rank', 'Feature', 'Mean_SHAP']]
      .to_string(index=False))

### 📝 SHAP Business Interpretation

> Fill in after running with your actual top features.
> Template below — replace X with real values:

**Top Features Analysis:**

**1. [Top Feature] — SHAP value: X.XXXX**
[Explain what this feature is and why it drives claims]
Business implication: [what ACIS should do with this]

**2. [Feature 2] — SHAP value: X.XXXX**
[Explain]

**3. [Feature 3] — SHAP value: X.XXXX**
[Explain]

**Validation of Hypothesis Testing:**
- PostalCode ranked #[X] in SHAP importance
  → [confirms/does not confirm] H2 and H3 findings
- Province ranked #[X] in SHAP importance
  → [confirms/does not confirm] H1 findings
- Gender ranked #[X] or not in top 10
  → [confirms/does not confirm] H4 finding

In [ ]:
# ─── RISK-BASED PRICING DEMONSTRATION ───────────────────────
print('=' * 55)
print('RISK-BASED PREMIUM CALCULATOR DEMO')
print('=' * 55)

# Use best models to price example policies
best_sev_model  = severity_results[best_sev]['model']
best_prob_model = prob_results[best_prob]['model']

# Take 5 random test policies and price them
sample_policies = X_test_p.iloc[:5].copy()

# Get claim probabilities from best classifier
p_claims = best_prob_model.predict_proba(
    sample_policies
)[:, 1]

# Get severity predictions from best regressor
# Need to match features — use first 5 from severity test set
sample_sev = X_test_s.iloc[:5].copy()
log_severities = best_sev_model.predict(sample_sev)
# Convert from log scale back to Rand
severities = np.expm1(log_severities)

print(f"\n{'Policy':<8} {'P(claim)':<12} "
      f"{'Pred Severity':<18} {'Exp Loss':<14} "
      f"{'Rec Premium':<14}")
print('-' * 65)

for i, (p, sev) in enumerate(zip(p_claims, severities)):
    result = calculate_premium(p, sev)
    print(
        f"  {i+1:<6} "
        f"{result['p_claim']:<12.4f} "
        f"R {result['predicted_severity']:<15,.2f} "
        f"R {result['expected_loss']:<12,.2f} "
        f"R {result['recommended_premium']:<12,.2f}"
    )

print('\nFormula: Premium = (P(claim) × Severity) × 1.25')
print('         (25% loading covers expenses + profit)')

---
## 5. Model Summary & Recommendations

### Severity Model Comparison

| Model | RMSE | R² | Verdict |
|---|---|---|---|
| Linear Regression | [fill] | [fill] | Baseline |
| Random Forest | [fill] | [fill] | [fill] |
| XGBoost | [fill] | [fill] | Best/Runner-up |

### Probability Model Comparison

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Logistic Regression | [fill] | [fill] | [fill] | [fill] |
| Random Forest | [fill] | [fill] | [fill] | [fill] |
| XGBoost | [fill] | [fill] | [fill] | [fill] |

### Top SHAP Features
1. [fill]
2. [fill]
3. [fill]

### Pricing Recommendations
Based on model outputs:
- High-risk policies: premium should be R[X]–R[X]
- Low-risk policies: premium could be reduced to R[X]
- Province and PostalCode are the strongest drivers